In [4]:
%%writefile sumMatrix2D.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <sys/time.h>

#define CHECK(call)                                                            \
do {                                                                           \
    const cudaError_t error = call;                                            \
    if (error != cudaSuccess)                                                  \
    {                                                                          \
        fprintf(stderr, "Error: %s:%d, ", __FILE__, __LINE__);                 \
        fprintf(stderr, "code: %d, reason: %s\n", error,                       \
                cudaGetErrorString(error));                                    \
        exit(1);                                                               \
    }                                                                          \
} while (0)

inline double seconds()
{
    struct timeval tp;
    struct timezone tzp;
    gettimeofday(&tp, &tzp);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1e-6;
}

// ---------------- INITIALIZE INTEGER DATA ----------------
void initialData(int *ip, const int size)
{
    for (int i = 0; i < size; i++)
        ip[i] = rand() % 100;  // integer values only
}

// ---------------- HOST MATRIX SUM ----------------
void sumMatrixOnHost(int *A, int *B, int *C, int nx, int ny)
{
    for (int iy = 0; iy < ny; iy++)
        for (int ix = 0; ix < nx; ix++)
        {
            int idx = iy * nx + ix;
            C[idx] = A[idx] + B[idx];
        }
}

// ---------------- COMPARE GPU + CPU RESULTS ----------------
void checkResult(int *hostRef, int *gpuRef, int N)
{
    for (int i = 0; i < N; i++)
    {
        if (hostRef[i] != gpuRef[i])
        {
            printf("Mismatch at %d: host %d gpu %d\n",
                    i, hostRef[i], gpuRef[i]);
            return;
        }
    }
    printf("Arrays match!\n");
}

// ---------------- GPU KERNEL ----------------
__global__ void sumMatrixOnGPU2D(int *A, int *B, int *C, int NX, int NY)
{
    unsigned int ix = blockIdx.x * blockDim.x + threadIdx.x;
    unsigned int iy = blockIdx.y * blockDim.y + threadIdx.y;

    if (ix < NX && iy < NY)
    {
        int idx = iy * NX + ix;
        C[idx] = A[idx] + B[idx];
    }
}

// ---------------- MAIN FUNCTION ----------------
int main()
{
    int nx = 1 << 12;  // 4096
    int ny = 1 << 12;  // 4096
    int nxy = nx * ny;
    int nBytes = nxy * sizeof(int);

    printf("Matrix Summation using 2D Grid + 2D Block\n");
    printf("Matrix size: %d x %d = %d elements\n", nx, ny, nxy);

    // Allocate host memory
    int *h_A = (int*)malloc(nBytes);
    int *h_B = (int*)malloc(nBytes);
    int *hostRef = (int*)malloc(nBytes);
    int *gpuRef  = (int*)malloc(nBytes);

    // Initialize matrices
    double start = seconds();
    initialData(h_A, nxy);
    initialData(h_B, nxy);
    printf("Data initialization: %f sec\n", seconds() - start);

    // CPU sum
    start = seconds();
    sumMatrixOnHost(h_A, h_B, hostRef, nx, ny);
    printf("CPU execution time: %f sec\n", seconds() - start);

    // GPU memory allocation
    int *d_A, *d_B, *d_C;
    CHECK(cudaMalloc((void**)&d_A, nBytes));
    CHECK(cudaMalloc((void**)&d_B, nBytes));
    CHECK(cudaMalloc((void**)&d_C, nBytes));

    CHECK(cudaMemcpy(d_A, h_A, nBytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_B, h_B, nBytes, cudaMemcpyHostToDevice));

    // ------- FIXED BLOCK SIZE -------
    dim3 block(16, 16);  // SAFE SIZE (256 threads/block)
    dim3 grid((nx + block.x - 1) / block.x,
              (ny + block.y - 1) / block.y);

    printf("Launching kernel <<<(%d,%d), (%d,%d)>>>\n",
           grid.x, grid.y, block.x, block.y);

    // GPU kernel execution
    start = seconds();
    sumMatrixOnGPU2D<<<grid, block>>>(d_A, d_B, d_C, nx, ny);

    // Check kernel error
    CHECK(cudaGetLastError());       // <---- IMPORTANT FIX
    CHECK(cudaDeviceSynchronize());
    printf("GPU execution time: %f sec\n", seconds() - start);

    // Copy back result
    CHECK(cudaMemcpy(gpuRef, d_C, nBytes, cudaMemcpyDeviceToHost));

    // Compare CPU + GPU results
    checkResult(hostRef, gpuRef, nxy);

    // Free memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(hostRef);
    free(gpuRef);

    CHECK(cudaDeviceReset());
    return 0;
}

Overwriting sumMatrix2D.cu


In [5]:
!nvcc -arch=sm_70 sumMatrix2D.cu -o sumMatrix2D
!./sumMatrix2D

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Matrix Summation using 2D Grid + 2D Block
Matrix size: 4096 x 4096 = 16777216 elements
Data initialization: 0.872869 sec
CPU execution time: 0.084321 sec
Launching kernel <<<(256,256), (16,16)>>>
GPU execution time: 0.001631 sec
Arrays match!
